In [1]:
from pprint import pprint

from openai import OpenAI
from openai.types.responses import ResponseFunctionToolCall, ResponseOutputMessage

from chat.func import OpenAIService
from chat.config import MODEL
from chat.tool import assemble_pc
from prompt.system import CONSULTANT_SYSTEM_PROMPT, EXTRACT_FILTERS_SYSTEM_PROMPT


client = OpenAI()
ai_service = OpenAIService(
    model=MODEL,
    system_prompt=CONSULTANT_SYSTEM_PROMPT,
    client=client,
    tools=[assemble_pc]
)

ai_filter_service = OpenAIService(
    model=MODEL,
    system_prompt=EXTRACT_FILTERS_SYSTEM_PROMPT,
    client=client,
)


history = []

In [7]:
response = None


def assemble_pc_tool(service: OpenAIService, history: list[dict[str]]):

    user_replica = service.create_user_replica(
        "Get absolutely all filters from previous messages."
    )
    response = service.request(user_replica, history)

    return response


def func(msg: str):
    global history
    global response

    replica = ai_service.create_user_replica(msg)
    response = ai_service.request(replica, history)



    history += [replica]

    # if response.output[0].type == "message"

    print(1)

    print(response.output)


    if len(response.output) > 1:
        print("ERROR\nLEN response.output > 1")

    output_item = response.output[0]
    if output_item.type == "message":
        # "role", "content", "type"

        output_item = output_item.to_dict()
        del output_item["id"]
        del output_item["status"]
        output_item["content"] = output_item["content"][0]["text"]
    elif output_item.type == "function_call":
        
        # тут дістаються фільтри:
        if output_item.name == "assemble_pc":
            call_function_result = assemble_pc_tool(ai_filter_service, history)
            print(call_function_result)
        
        
        pass

    history.append(output_item)


    print(2)


In [ ]:
response.output[0].to_dict()

In [8]:
func("збери ПК до 15 тисяч, на інтел, з відеокартою rtx 2060 і 16 gb ОЗУ. більше нічого не питий, а зразу збери ПК")

1
[ResponseOutputMessage(id='msg_681ddc9e2f048191a5b0ab8514a1c729061fc18a2a81ea7e', content=[ResponseOutputText(annotations=[], text='Які ще компоненти ти хотів би включити? Наприклад, обсяг жорсткого диска (SSD або HDD), материнська плата, блок живлення, корпус, чи є якісь уподобання щодо брендів?', type='output_text')], role='assistant', status='completed', type='message')]
2


In [9]:
func("просто збери ПК і більше не питай ніого")

1
[ResponseFunctionToolCall(arguments='{}', call_id='call_jQzWjZzfRmP9UchPYKeOBDII', name='assemble_pc', type='function_call', id='fc_681ddca2701c81919a04b739d28881ca01bc3854392d1fd8', status='completed')]
Response(id='resp_681ddca30a1481919de3723487a1fdc8045f0adbfe8478f5', created_at=1746787491.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseOutputMessage(id='msg_681ddca368f8819186f2b2a659a9235e045f0adbfe8478f5', content=[ResponseOutputText(annotations=[], text='```json\n{\n  "budget": 15000,\n  "cpu": {\n    "type": "Intel",\n    "cores": "not specified"\n  },\n  "gpu": {\n    "model": "RTX 2060",\n    "memory": "not specified"\n  },\n  "ram": {\n    "size": 16,\n    "type": "not specified"\n  },\n  "storage": {\n    "type": "not specified",\n    "capacity": "not specified"\n  },\n  "case": {\n    "format": "not specified"\n  },\n  "tasks": "not specified"\n}\n```', type='output_text')], role='

In [ ]:
func("привіт, хто ти?")

In [ ]:
func("хочу ПК")

In [ ]:
func("10 000 гривень")

In [ ]:
func("хочу грати майнкрафт та терарію")

In [ ]:
func("я не розбираюся у компонентах")

In [ ]:
func("нічого не потрібно")

In [ ]:
func("немає")

In [ ]:
func("вибери просто оптимальні компоненти")

In [ ]:
func("просто збери мені ПК")

In [ ]:
func("тепер збер ПК")

In [ ]:
print(*history, sep="\n\n")